# Step 2: Curating the Lake — Parquet & Partitioning

Now we turn the raw CSV into something an engine can actually exploit: **Parquet**, a columnar, compressed file format with an embedded schema. We use DuckDB to do the conversion — no separate ETL framework needed.

We also **partition** the output by year and month. A partition is nothing magical: it's just a folder. `lake/verkauf/jahr=2026/monat=08/data.parquet` tells any engine "everything in here is from August 2026" purely through the folder name — no database, no catalog required.

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

In [ ]:
import duckdb

con = duckdb.connect()
con.sql("SELECT * FROM read_csv_auto('lake/raw/sales.csv') LIMIT 5").show()

## Convert to partitioned Parquet

`COPY ... TO ... (FORMAT PARQUET, PARTITION_BY (...))` reads the CSV, derives `jahr` (year) and `monat` (month) from the date, and writes one Parquet file per partition folder — all in a single statement.

In [ ]:
con.sql("""
    COPY (
        SELECT
            *,
            EXTRACT(year FROM date)  AS jahr,
            EXTRACT(month FROM date) AS monat
        FROM read_csv_auto('lake/raw/sales.csv')
    ) TO 'lake/verkauf' (
        FORMAT PARQUET,
        PARTITION_BY (jahr, monat),
        OVERWRITE_OR_IGNORE 1
    )
""")
print("Done.")

## Inspect the partition layout

This is a plain folder tree — open it in the file explorer, or list it here.

In [ ]:
for path in sorted(Path("lake/verkauf").rglob("*.parquet"))[:12]:
    print(path)

## The size comparison (CSV vs. Parquet)

Expect a 5-10x reduction: columnar layout compresses repeated values (region, product) far better than row-based text.

In [ ]:
def dir_size_mb(path: str) -> float:
    total_bytes = sum(
        f.stat().st_size for f in Path(path).rglob("*") if f.is_file()
    )
    return total_bytes / (1024 * 1024)

csv_mb = dir_size_mb("lake/raw")
parquet_mb = dir_size_mb("lake/verkauf")

print(f"CSV (raw):          {csv_mb:8.1f} MB")
print(f"Parquet (curated):  {parquet_mb:8.1f} MB")
print(f"Reduction factor:   {csv_mb / parquet_mb:8.1f}x")

Two things happened here, and both are visible without touching a database:

1. **Compression** — the folder on disk shrank by 5-10x.
2. **Partitioning** — the data is now organized by `jahr=`/`monat=` folders, which the next notebook's query engine can use to skip work entirely.